## One-Hot Encoding

In [ ]:
# ========================================================
# FILE 1: churn_onehot_only.py
# Method: One-Hot Encoding ONLY (for categorical columns)
# No text processing (CustomerFeedback column is ignored/dropped)
# Beginner-friendly - Minimal steps with detailed comments
# ========================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ====================== STEP 1: Generate 150 Records ======================
print("=== STEP 1: Generating 150 synthetic customer records ===")
np.random.seed(42)

n_samples = 150

data = {
    'Gender': np.random.choice(['Male', 'Female'], n_samples),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.55, 0.30, 0.15]),
    'PaymentMethod': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples),
    'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.35, 0.50, 0.15]),
    'Tenure': np.random.randint(1, 73, n_samples),
    'MonthlyCharges': np.round(np.random.uniform(20, 120, n_samples), 2),
    'CustomerFeedback': np.random.choice([
        "Great service", "Poor connection", "Excellent support", "Billing issues",
        "Love the pricing", "Slow internet", "Happy overall", "Too expensive"
    ], n_samples)
}

df = pd.DataFrame(data)

# Create target (Churn)
high_churn = (df['Tenure'] < 12) | (df['MonthlyCharges'] > 80)
df['Churn'] = np.where(high_churn,
                       np.random.choice(['Yes', 'No'], n_samples, p=[0.75, 0.25]),
                       np.random.choice(['Yes', 'No'], n_samples, p=[0.20, 0.80]))

print(f"Total records: {df.shape[0]}")
print("Sample data:\n", df.head(5))
print("\nChurn distribution:\n", df['Churn'].value_counts())

# ====================== STEP 2: Prepare Features (X) and Target (y) ======================
X = df.drop('Churn', axis=1)   # All columns except target
y = df['Churn']

categorical_features = ['Gender', 'Contract', 'PaymentMethod', 'InternetService']
numerical_features   = ['Tenure', 'MonthlyCharges']
# Note: CustomerFeedback is automatically dropped because we don't include it below

# ====================== STEP 3: Create Preprocessor with OneHotEncoder ======================
print("\n=== STEP 3: Creating preprocessor with One-Hot Encoding ===")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),                    # Scale numbers
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)  # One-Hot Encoding
    ],
    remainder='drop'   # This drops CustomerFeedback column
)

# ====================== STEP 4: Create Full Pipeline ======================
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# ====================== STEP 5: Train-Test Split (80% train, 20% test) ======================
print("\n=== STEP 5: Splitting data into train and test ===")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training records: {X_train.shape[0]} | Test records: {X_test.shape[0]}")

# ====================== STEP 6: Train the Model ======================
print("\n=== STEP 6: Training the model ===")
pipeline.fit(X_train, y_train)

# ====================== STEP 7: Evaluate on Test Data ======================
print("\n=== STEP 7: Checking accuracy on test data ===")
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ====================== STEP 8: Predict on New Data (Future Use) ======================
print("\n=== STEP 8: Predicting on new customer data ===")
new_data = pd.DataFrame({
    'Gender': ['Female', 'Male'],
    'Contract': ['Two year', 'Month-to-month'],
    'PaymentMethod': ['Credit card', 'Electronic check'],
    'InternetService': ['DSL', 'Fiber optic'],
    'Tenure': [48, 5],
    'MonthlyCharges': [55.75, 95.50],
    'CustomerFeedback': ["Great service", "Poor connection"]   # ignored automatically
})

predictions = pipeline.predict(new_data)
probabilities = pipeline.predict_proba(new_data)

print("New Data:")
print(new_data)
print("\nPredictions:")
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    print(f"Customer {i+1}: Churn = {pred} | Probability = {prob[1]:.2%}")

print("\n🎉 One-Hot Encoding Only script complete!")

## Bag of Words

In [ ]:
# ========================================================
# FILE 2: churn_onehot_bow.py
# Method: One-Hot Encoding + BOW (Bag of Words)
# Uses CountVectorizer on CustomerFeedback text
# Beginner-friendly - Minimal steps with detailed comments
# ========================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ====================== STEP 1: Generate 150 Records ======================
print("=== STEP 1: Generating 150 synthetic customer records ===")
np.random.seed(42)

n_samples = 150

data = {
    'Gender': np.random.choice(['Male', 'Female'], n_samples),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.55, 0.30, 0.15]),
    'PaymentMethod': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples),
    'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.35, 0.50, 0.15]),
    'Tenure': np.random.randint(1, 73, n_samples),
    'MonthlyCharges': np.round(np.random.uniform(20, 120, n_samples), 2),
    'CustomerFeedback': np.random.choice([
        "Great service and fast speeds",
        "Poor network connection keeps dropping",
        "Excellent customer support",
        "Billing errors every month",
        "Love the pricing and value",
        "Slow internet during evenings",
        "Happy with the overall experience",
        "Contract renewal is too expensive"
    ], n_samples)
}

df = pd.DataFrame(data)

# Create target (Churn)
high_churn = (df['Tenure'] < 12) | (df['MonthlyCharges'] > 80)
df['Churn'] = np.where(high_churn,
                       np.random.choice(['Yes', 'No'], n_samples, p=[0.75, 0.25]),
                       np.random.choice(['Yes', 'No'], n_samples, p=[0.20, 0.80]))

print(f"Total records: {df.shape[0]}")
print("Sample data:\n", df.head(5))
print("\nChurn distribution:\n", df['Churn'].value_counts())

# ====================== STEP 2: Prepare Features (X) and Target (y) ======================
X = df.drop('Churn', axis=1)   # All columns including text
y = df['Churn']

categorical_features = ['Gender', 'Contract', 'PaymentMethod', 'InternetService']
numerical_features   = ['Tenure', 'MonthlyCharges']
text_feature         = 'CustomerFeedback'

# ====================== STEP 3: Create Preprocessor with OneHot + BOW ======================
print("\n=== STEP 3: Creating preprocessor with One-Hot Encoding + BOW ===")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('text', CountVectorizer(max_features=50, stop_words='english'), text_feature)   # BOW here
    ],
    remainder='drop'
)

# ====================== STEP 4: Create Full Pipeline ======================
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# ====================== STEP 5: Train-Test Split ======================
print("\n=== STEP 5: Splitting data into train and test ===")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training records: {X_train.shape[0]} | Test records: {X_test.shape[0]}")

# ====================== STEP 6: Train the Model ======================
print("\n=== STEP 6: Training the model ===")
pipeline.fit(X_train, y_train)

# ====================== STEP 7: Evaluate on Test Data ======================
print("\n=== STEP 7: Checking accuracy on test data ===")
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ====================== STEP 8: Predict on New Data ======================
print("\n=== STEP 8: Predicting on new customer data ===")
new_data = pd.DataFrame({
    'Gender': ['Female', 'Male'],
    'Contract': ['Two year', 'Month-to-month'],
    'PaymentMethod': ['Credit card', 'Electronic check'],
    'InternetService': ['DSL', 'Fiber optic'],
    'Tenure': [48, 5],
    'MonthlyCharges': [55.75, 95.50],
    'CustomerFeedback': ["Great service and fast speeds", "Network keeps dropping"]
})

predictions = pipeline.predict(new_data)
probabilities = pipeline.predict_proba(new_data)

print("New Data:")
print(new_data)
print("\nPredictions:")
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    print(f"Customer {i+1}: Churn = {pred} | Probability = {prob[1]:.2%}")

print("\n🎉 One-Hot + BOW script complete!")

## TF-IDF

In [ ]:
# ========================================================
# FILE 3: churn_onehot_tfidf.py
# Method: One-Hot Encoding + TF-IDF
# Uses TfidfVectorizer on CustomerFeedback text
# Beginner-friendly - Minimal steps with detailed comments
# ========================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ====================== STEP 1: Generate 150 Records ======================
print("=== STEP 1: Generating 150 synthetic customer records ===")
np.random.seed(42)

n_samples = 150

data = {
    'Gender': np.random.choice(['Male', 'Female'], n_samples),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.55, 0.30, 0.15]),
    'PaymentMethod': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples),
    'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.35, 0.50, 0.15]),
    'Tenure': np.random.randint(1, 73, n_samples),
    'MonthlyCharges': np.round(np.random.uniform(20, 120, n_samples), 2),
    'CustomerFeedback': np.random.choice([
        "Great service and fast speeds",
        "Poor network connection keeps dropping",
        "Excellent customer support",
        "Billing errors every month",
        "Love the pricing and value",
        "Slow internet during evenings",
        "Happy with the overall experience",
        "Contract renewal is too expensive"
    ], n_samples)
}

df = pd.DataFrame(data)

# Create target (Churn)
high_churn = (df['Tenure'] < 12) | (df['MonthlyCharges'] > 80)
df['Churn'] = np.where(high_churn,
                       np.random.choice(['Yes', 'No'], n_samples, p=[0.75, 0.25]),
                       np.random.choice(['Yes', 'No'], n_samples, p=[0.20, 0.80]))

print(f"Total records: {df.shape[0]}")
print("Sample data:\n", df.head(5))
print("\nChurn distribution:\n", df['Churn'].value_counts())

# ====================== STEP 2: Prepare Features (X) and Target (y) ======================
X = df.drop('Churn', axis=1)   # All columns including text
y = df['Churn']

categorical_features = ['Gender', 'Contract', 'PaymentMethod', 'InternetService']
numerical_features   = ['Tenure', 'MonthlyCharges']
text_feature         = 'CustomerFeedback'

# ====================== STEP 3: Create Preprocessor with OneHot + TF-IDF ======================
print("\n=== STEP 3: Creating preprocessor with One-Hot Encoding + TF-IDF ===")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('text', TfidfVectorizer(max_features=50, stop_words='english'), text_feature)   # TF-IDF here
    ],
    remainder='drop'
)

# ====================== STEP 4: Create Full Pipeline ======================
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# ====================== STEP 5: Train-Test Split ======================
print("\n=== STEP 5: Splitting data into train and test ===")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training records: {X_train.shape[0]} | Test records: {X_test.shape[0]}")

# ====================== STEP 6: Train the Model ======================
print("\n=== STEP 6: Training the model ===")
pipeline.fit(X_train, y_train)

# ====================== STEP 7: Evaluate on Test Data ======================
print("\n=== STEP 7: Checking accuracy on test data ===")
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ====================== STEP 8: Predict on New Data ======================
print("\n=== STEP 8: Predicting on new customer data ===")
new_data = pd.DataFrame({
    'Gender': ['Female', 'Male'],
    'Contract': ['Two year', 'Month-to-month'],
    'PaymentMethod': ['Credit card', 'Electronic check'],
    'InternetService': ['DSL', 'Fiber optic'],
    'Tenure': [48, 5],
    'MonthlyCharges': [55.75, 95.50],
    'CustomerFeedback': ["Great service and fast speeds", "Network keeps dropping"]
})

predictions = pipeline.predict(new_data)
probabilities = pipeline.predict_proba(new_data)

print("New Data:")
print(new_data)
print("\nPredictions:")
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    print(f"Customer {i+1}: Churn = {pred} | Probability = {prob[1]:.2%}")

print("\n🎉 One-Hot + TF-IDF script complete!")

## OHE, BoW, TF-IDF Combined Testing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# ========================================================
# END-TO-END CUSTOMER CHURN PREDICTION PROJECT
# ========================================================
# Problem: Telecom company wants to predict if a customer will churn (Yes/No)
# Features: Categorical + Numerical + TEXT feedback (CustomerFeedback)
#
# Techniques Compared (All 3 methods):
#   1. OneHotEncoding only  (for categorical columns)
#   2. OneHotEncoding + BOW (Bag of Words using CountVectorizer on text)
#   3. OneHotEncoding + TF-IDF (using TfidfVectorizer on text)
#
# We train all three pipelines, compare accuracy on test set,
# automatically select the BEST performing method,
# and use that best model for any future / new dataset.
# Data: 150 realistic synthetic records
# ========================================================

# ====================== STEP 1: Generate 150 Records ======================
print("=== STEP 1: Creating 150 synthetic customer records ===")

np.random.seed(42)

n_samples = 150

data = {
    'Gender': np.random.choice(['Male', 'Female'], n_samples),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.55, 0.30, 0.15]),
    'PaymentMethod': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples),
    'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.35, 0.50, 0.15]),
    'Tenure': np.random.randint(1, 73, n_samples),
    'MonthlyCharges': np.round(np.random.uniform(20, 120, n_samples), 2),
}

# TEXT column for BOW and TF-IDF
feedback_options = [
    "Great service and fast speeds",
    "Poor network connection keeps dropping",
    "Excellent customer support",
    "Billing errors every month",
    "Love the pricing and value",
    "Slow internet during peak hours",
    "Happy with the overall experience",
    "Contract is too expensive",
    "Frequent network outages",
    "Very satisfied with fiber optic",
    "Support team never responds",
    "Switched due to better offers",
    "Reliable connection and good speed",
    "Too many hidden charges",
    "Best internet service provider"
]
data['CustomerFeedback'] = np.random.choice(feedback_options, n_samples)

df = pd.DataFrame(data)

# Create target variable Churn (realistic)
high_churn = (df['Tenure'] < 12) | (df['MonthlyCharges'] > 80)
df['Churn'] = np.where(
    high_churn,
    np.random.choice(['Yes', 'No'], n_samples, p=[0.75, 0.25]),
    np.random.choice(['Yes', 'No'], n_samples, p=[0.20, 0.80])
)

print(f"Total records: {df.shape[0]}")
print(df.head())
print("\nChurn distribution:\n", df['Churn'].value_counts())

# ====================== STEP 2: Define Features ======================
X = df.drop('Churn', axis=1)
y = df['Churn']

categorical_features = ['Gender', 'Contract', 'PaymentMethod', 'InternetService']
numerical_features   = ['Tenure', 'MonthlyCharges']
text_feature         = 'CustomerFeedback'

# ====================== STEP 3: Helper Function to Create Pipelines ======================
def create_pipeline(method):
    """Create pipeline for different encoding methods"""
    if method == 'onehot_only':
        # Only OneHotEncoding + scaling (no text vectorization)
        text_transformer = 'drop'
        name = "OneHot Encoding Only"
    elif method == 'bow':
        text_transformer = CountVectorizer(max_features=50, stop_words='english')
        name = "OneHot + BOW (CountVectorizer)"
    else:  # tfidf
        text_transformer = TfidfVectorizer(max_features=50, stop_words='english')
        name = "OneHot + TF-IDF"
    
    transformers = [
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
    
    if method != 'onehot_only':
        transformers.append(('text', text_transformer, text_feature))
    
    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    
    return pipeline, name

# Create all 3 pipelines
pipelines = {}
methods = ['onehot_only', 'bow', 'tfidf']

for m in methods:
    pipe, name = create_pipeline(m)
    pipelines[m] = (pipe, name)

# ====================== STEP 4: Train-Test Split ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\n=== STEP 4: Training on {X_train.shape[0]} records | Testing on {X_test.shape[0]} ===")

# ====================== STEP 5: Train & Compare All 3 Methods ======================
print("\n=== STEP 5: Training and Comparing All 3 Methods ===")

results = {}

for m in methods:
    pipe, name = pipelines[m]
    print(f"Training {name} ...")
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[m] = (pipe, name, acc, y_pred)
    print(f"   → {name} Accuracy: {acc:.2%}")

# Find the BEST method
best_method = max(results, key=lambda k: results[k][2])
best_pipe, best_name, best_acc, best_pred = results[best_method]

print(f"\n✅ BEST METHOD SELECTED: {best_name}")
print(f"   Best Accuracy: {best_acc:.2%}")

# Detailed performance of best model
print("\nClassification Report (Best Model):")
print(classification_report(y_test, best_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, best_pred))

# ====================== STEP 6: Save the Best Model ======================
joblib.dump(best_pipe, 'best_churn_model_onehot_bow_tfidf.pkl')
print(f"\n✅ Best model saved as 'best_churn_model_onehot_bow_tfidf.pkl'")

# ====================== STEP 7: Predict on New/Future Dataset ======================
print("\n=== STEP 7: Prediction on New Customer Data (Future Use) ===")

# Sample new customers - replace with your real data
new_customers = pd.DataFrame({
    'Gender': ['Female', 'Male', 'Female'],
    'Contract': ['Two year', 'Month-to-month', 'One year'],
    'PaymentMethod': ['Credit card', 'Electronic check', 'Bank transfer'],
    'InternetService': ['DSL', 'Fiber optic', 'No'],
    'Tenure': [48, 5, 24],
    'MonthlyCharges': [55.75, 95.50, 65.00],
    'CustomerFeedback': [
        "Excellent service and very fast speeds",
        "Network keeps dropping and poor support",
        "Good connection but billing is confusing"
    ]
})

# Use the BEST model (automatically applies correct OneHot + BOW/TF-IDF or only OneHot)
predictions = best_pipe.predict(new_customers)
probabilities = best_pipe.predict_proba(new_customers)

print("New Customer Data:")
print(new_customers)
print("\n🔮 Predictions using BEST method:")
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    print(f"Customer {i+1}: Predicted Churn = {pred} | Churn Probability = {prob[1]:.2%}")

# ====================== STEP 8: Production Usage on Any Future Dataset ======================
print("\n=== STEP 8: How to Use on Any Future Dataset ===")
print("""
# In production or for any new CSV file:
loaded_model = joblib.load('best_churn_model_onehot_bow_tfidf.pkl')

# future_df = pd.read_csv('new_customers.csv')   # Must have same columns
# predictions = loaded_model.predict(future_df)
# future_df['Predicted_Churn'] = predictions
# future_df.to_csv('churn_predictions.csv', index=False)
""")

print("\n🎉 PROJECT COMPLETE!")
print("All 3 methods (OneHot Only, OneHot+BOW, OneHot+TF-IDF) were trained and compared.")
print(f"Best performing method: {best_name}")
print("The saved model will automatically use the winning technique for any future predictions.")